In [1]:
!pip install albumentations opencv-python lxml scikit-learn  -q

# Logging

In [2]:
import os
import cv2
import copy
import numpy as np
import xml.etree.ElementTree as ET
from tqdm import tqdm
from pathlib import Path

import os
import shutil
import cv2
import copy
from lxml import etree
from collections import defaultdict
from sklearn.model_selection import train_test_split
import albumentations as A

import os
import cv2
import copy
import shutil
import albumentations as A
from lxml import etree
from tqdm import tqdm

import os
import csv
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from collections import Counter
from typing import List, Tuple
from tabulate import tabulate

from torchmetrics.detection.mean_ap import MeanAveragePrecision
from torchvision.ops import box_iou
from sklearn.metrics import confusion_matrix

import logging
import sys
import os
from datetime import datetime
import random
import numpy as np
from collections import defaultdict
from tabulate import tabulate

In [3]:

def setup_logger(log_dir="logs", log_name="train"):
    os.makedirs(log_dir, exist_ok=True)

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    log_file = os.path.join(log_dir, f"{log_name}_{timestamp}.log")

    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s | %(levelname)s | %(message)s",
        handlers=[
            logging.FileHandler(log_file),
            logging.StreamHandler(sys.stdout)
        ]
    )

    return log_file

log_file = setup_logger()

import torch
if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

# Agumnetation

In [4]:
# ============================================================
# HELPER FUNCTIONS (FIXED & IMPLEMENTED)
# ============================================================

# Helper functions
def compute_stats(xml_dir):
        """Compute statistics of classes in XML annotations"""
        stats = {}
        xml_files = [f for f in os.listdir(xml_dir) if f.endswith('.xml') and "_aug" not in f]
        
        for xml_file in xml_files:
            xml_path = os.path.join(xml_dir, xml_file)
            try:
                tree = ET.parse(xml_path)
                root = tree.getroot()
                
                for obj in root.findall('object'):
                    name = obj.find('name').text
                    stats[name] = stats.get(name, 0) + 1
            except Exception as e:
                print(f"⚠️ Error reading {xml_file}: {e}")
        
        return stats
    
def read_xml(xml_path):
        """Read XML annotation file using xml.etree.ElementTree"""
        tree = ET.parse(xml_path)
        root = tree.getroot()
        
        boxes = []
        labels = []
        
        for obj in root.findall('object'):
            name = obj.find('name')
            if name is not None:
                label = name.text
                bndbox = obj.find('bndbox')
                
                if bndbox is not None:
                    try:
                        xmin = float(bndbox.find('xmin').text)
                        ymin = float(bndbox.find('ymin').text)
                        xmax = float(bndbox.find('xmax').text)
                        ymax = float(bndbox.find('ymax').text)
                        
                        # Ensure valid bounding box
                        if xmax > xmin and ymax > ymin:
                            boxes.append([xmin, ymin, xmax, ymax])
                            labels.append(label)
                        else:
                            print(f"⚠️ Invalid bbox in {xml_path}: ({xmin}, {ymin}, {xmax}, {ymax})")
                    except (ValueError, AttributeError) as e:
                        print(f"⚠️ Error parsing bbox in {xml_path}: {e}")
        
        return tree, root, boxes, labels
    
def update_xml_for_augmentation(xml_path, boxes, labels, img_name, img_width, img_height):
        """Create updated XML for augmented image"""
        # Parse original XML to get base structure
        tree = ET.parse(xml_path)
        root = tree.getroot()
        
        # Update filename
        filename_elem = root.find('filename')
        if filename_elem is not None:
            filename_elem.text = img_name
        else:
            filename_elem = ET.SubElement(root, 'filename')
            filename_elem.text = img_name
        
        # Update size
        size_elem = root.find('size')
        if size_elem is None:
            size_elem = ET.SubElement(root, 'size')
        
        # Clear old size elements
        for elem in list(size_elem):
            size_elem.remove(elem)
        
        ET.SubElement(size_elem, 'width').text = str(int(img_width))
        ET.SubElement(size_elem, 'height').text = str(int(img_height))
        ET.SubElement(size_elem, 'depth').text = '3'
        
        # Remove all existing object elements
        objects = root.findall('object')
        for obj in objects:
            root.remove(obj)
        
        # Add new object elements
        for i, (bbox, label) in enumerate(zip(boxes, labels)):
            if label is None:
                continue
                
            obj_elem = ET.SubElement(root, 'object')
            
            # Add class name
            name_elem = ET.SubElement(obj_elem, 'name')
            name_elem.text = str(label)
            
            # Add other required fields
            ET.SubElement(obj_elem, 'pose').text = 'Unspecified'
            ET.SubElement(obj_elem, 'truncated').text = '0'
            ET.SubElement(obj_elem, 'difficult').text = '0'
            
            # Add bounding box
            bndbox_elem = ET.SubElement(obj_elem, 'bndbox')
            
            # Extract coordinates
            xmin, ymin, xmax, ymax = bbox
            
            # Ensure coordinates are valid
            xmin = max(0, float(xmin))
            ymin = max(0, float(ymin))
            xmax = min(img_width - 1, float(xmax))
            ymax = min(img_height - 1, float(ymax))
            
            # Ensure proper order
            if xmin > xmax:
                xmin, xmax = xmax, xmin
            if ymin > ymax:
                ymin, ymax = ymax, ymin
            
            # Skip invalid boxes
            if xmax <= xmin or ymax <= ymin:
                print(f"⚠️ Skipping invalid box in {img_name}: {bbox}")
                continue
            
            # Convert to integers
            xmin = int(round(xmin))
            ymin = int(round(ymin))
            xmax = int(round(xmax))
            ymax = int(round(ymax))
            
            ET.SubElement(bndbox_elem, 'xmin').text = str(xmin)
            ET.SubElement(bndbox_elem, 'ymin').text = str(ymin)
            ET.SubElement(bndbox_elem, 'xmax').text = str(xmax)
            ET.SubElement(bndbox_elem, 'ymax').text = str(ymax)
            
            print(f"  Box {i}: label={label}, bbox=[{xmin}, {ymin}, {xmax}, {ymax}]")
        
        return tree
    
# ============================================================
# SUMMARY & EXECUTION
# ============================================================

def print_summary(out_dir, before_train_stats):
    # Fixed paths to match your directory structure
    after_train = compute_stats(os.path.join(out_dir, "annotations"))
    
    print("\n📈 BALANCE SUMMARY")
    print(f"{'Class':<20}{'Train Pre':<12}{'Train Post':<12}")
    print("-" * 45)
    all_classes = sorted(before_train_stats.keys() | after_train.keys())
    for cls in all_classes:
        print(f"{cls:<20}{before_train_stats.get(cls,0):<12}{after_train.get(cls,0):<12}")

def prepare_and_augment_dataset(img_input, xml_input, output_base, target=2000):
    # Create paths
    train_img_path = os.path.join(output_base, "images")
    train_xml_path = os.path.join(output_base, "annotations")
    os.makedirs(train_img_path, exist_ok=True)
    os.makedirs(train_xml_path, exist_ok=True)

    # NEW: Copy original files to the augmentation workspace first!
    print("🚚 Copying original files to workspace...")
    for f in os.listdir(xml_input):
        if f.lower().endswith('.xml'):
            shutil.copy(os.path.join(xml_input, f), train_xml_path)
            # Find matching image (checking common extensions)
            base = os.path.splitext(f)[0]
            for ext in ['.jpg', '.jpeg', '.png', '.JPG']:
                img_file = base + ext
                if os.path.exists(os.path.join(img_input, img_file)):
                    shutil.copy(os.path.join(img_input, img_file), train_img_path)
                    break

    # Baseline stats
    before_stats = compute_stats(train_xml_path)
    if not before_stats:
        print("⚠️ Warning: No XML files found! Check your input paths.")
        return

    # Run Augmentation
    run_augmentation(train_img_path, train_xml_path, target_count=target)
     # Verify augmentations
    verify_saved_augmentation(
        train_img_path, 
        train_xml_path, 
        num_samples=100
    )
    # Final Report
    print_summary(output_base, before_stats)

In [5]:
# augmenter_fixed_complete.py
import os
import cv2
import copy
import numpy as np
import itertools
import shutil
from datetime import datetime
import xml.etree.ElementTree as ET
from collections import defaultdict
import random
from tqdm import tqdm
import warnings

warnings.filterwarnings("ignore", category=UserWarning)


class FixedAugmenter:
    """Complete fixed-augmentation class with high-diversity / low-cost transforms."""

    def __init__(self):
        self.transform_history = []

    # ---------- geometry helpers ----------
    @staticmethod
    def _clip_boxes(bboxes, w, h):
        out = []
        for bbox in bboxes:
            x1, y1, x2, y2 = bbox
            x1 = max(0, min(x1, w - 1))
            y1 = max(0, min(y1, h - 1))
            x2 = max(x1 + 1, min(x2, w))
            y2 = max(y1 + 1, min(y2, h))
            if x2 - x1 > 5 and y2 - y1 > 5:
                out.append([x1, y1, x2, y2])
        return out

    # ---------- individual transforms ----------
    def apply_horizontal_flip(self, image, bboxes):
        if image is None or len(image.shape) != 3:
            return image, bboxes
        w = image.shape[1]
        img = cv2.flip(image, 1)
        out = []
        for bbox in bboxes:
            if len(bbox) != 4:
                continue
            x1, y1, x2, y2 = bbox
            out.append([w - x2, y1, w - x1, y2])
        return img, self._clip_boxes(out, w, image.shape[0])

    def apply_vertical_flip(self, image, bboxes):
        if image is None or len(image.shape) != 3:
            return image, bboxes
        h = image.shape[0]
        img = cv2.flip(image, 0)
        out = []
        for bbox in bboxes:
            if len(bbox) != 4:
                continue
            x1, y1, x2, y2 = bbox
            out.append([x1, h - y2, x2, h - y1])
        return img, self._clip_boxes(out, image.shape[1], h)

    def apply_brightness_contrast(self, image):
        if image is None:
            return image
        alpha = np.random.uniform(0.8, 1.2)
        beta = np.random.randint(-30, 30)
        return np.clip(cv2.convertScaleAbs(image, alpha=alpha, beta=beta), 0, 255)

    def apply_noise(self, image):
        if image is None:
            return image
        noise = np.random.normal(0, 5, image.shape).astype(np.int16)
        return np.clip(image.astype(np.int16) + noise, 0, 255).astype(np.uint8)

    def apply_blur(self, image):
        if image is None:
            return image
        k = random.choice([3, 5])
        return cv2.GaussianBlur(image, (k, k), 0)

    def apply_hsv_shift(self, image):
        if image is None or len(image.shape) != 3:
            return image
        try:
            hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV).astype(np.float32)
            hsv[:, :, 0] = (hsv[:, :, 0] + np.random.uniform(-10, 10)) % 180
            hsv[:, :, 1] = np.clip(hsv[:, :, 1] * np.random.uniform(0.8, 1.2), 0, 255)
            hsv[:, :, 2] = np.clip(hsv[:, :, 2] * np.random.uniform(0.8, 1.2), 0, 255)
            return cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2BGR)
        except:
            return image

    def apply_rotation(self, image, bboxes):
        if image is None:
            return image, bboxes
        h, w = image.shape[:2]
        angle = np.random.uniform(-10, 10)
        M = cv2.getRotationMatrix2D((w / 2, h / 2), angle, 1.0)
        rotated = cv2.warpAffine(image, M, (w, h), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT)
        out = []
        for bbox in bboxes:
            if len(bbox) != 4:
                continue
            pts = np.array([[bbox[0], bbox[1]], [bbox[2], bbox[1]],
                            [bbox[2], bbox[3]], [bbox[0], bbox[3]]], dtype=np.float32)
            pts = (M @ np.column_stack([pts, np.ones(4)]).T).T
            xs, ys = pts[:, 0], pts[:, 1]
            x1, y1, x2, y2 = np.clip(xs.min(), 0, w), np.clip(ys.min(), 0, h), \
                np.clip(xs.max(), 0, w), np.clip(ys.max(), 0, h)
            if x2 - x1 > 5 and y2 - y1 > 5:
                out.append([x1, y1, x2, y2])
            else:
                out.append(bbox)
        return rotated, self._clip_boxes(out, w, h)

    def apply_scale(self, image, bboxes):
        if image is None:
            return image, bboxes
        h, w = image.shape[:2]
        scale = np.random.uniform(0.9, 1.1)
        new_w, new_h = int(w * scale), int(h * scale)
        if scale < 1.0:
            scaled = cv2.resize(image, (new_w, new_h))
            pad_x, pad_y = (w - new_w) // 2, (h - new_h) // 2
            canvas = np.zeros((h, w, 3), dtype=np.uint8)
            canvas[pad_y:pad_y + new_h, pad_x:pad_x + new_w] = scaled
            out = [[x * scale + pad_x, y * scale + pad_y, x2 * scale + pad_x, y2 * scale + pad_y]
                   for x, y, x2, y2 in bboxes]
            return canvas, self._clip_boxes(out, w, h)
        else:
            scaled = cv2.resize(image, (new_w, new_h))
            crop_x, crop_y = (new_w - w) // 2, (new_h - h) // 2
            cropped = scaled[crop_y:crop_y + h, crop_x:crop_x + w]
            out = [[max(0, x * scale - crop_x), max(0, y * scale - crop_y),
                    min(w, x2 * scale - crop_x), min(h, y2 * scale - crop_y)]
                   for x, y, x2, y2 in bboxes]
            return cropped, self._clip_boxes(out, w, h)

    def apply_translate(self, image, bboxes):
        if image is None:
            return image, bboxes
        h, w = image.shape[:2]
        dx = int(np.round(np.random.uniform(-0.06, 0.06) * w))
        dy = int(np.round(np.random.uniform(-0.06, 0.06) * h))
        M = np.float32([[1, 0, dx], [0, 1, dy]])
        shifted = cv2.warpAffine(image, M, (w, h), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT)
        out = [[x + dx, y + dy, x2 + dx, y2 + dy] for x, y, x2, y2 in bboxes]
        return shifted, self._clip_boxes(out, w, h)

    def apply_cutout(self, image):
        if image is None:
            return image
        h, w = image.shape[:2]
        side = int(0.06 * min(h, w))
        img = image.copy()
        for _ in range(3):
            x1 = np.random.randint(0, w - side)
            y1 = np.random.randint(0, h - side)
            img[y1:y1 + side, x1:x1 + side] = np.random.randint(0, 255, (side, side, 3), dtype=np.uint8)
        return img

    # ---------- safe orchestrator ----------
    def safe_augment(self, image, bboxes, labels):
        if image is None:
            return None, [], [], []
        aug_img = image.copy()
        aug_boxes = copy.deepcopy(bboxes)
        aug_labels = copy.deepcopy(labels)
        methods = []

        transforms = []
        if np.random.rand() > 0.5:
            transforms.append(('hflip', lambda img, boxes: self.apply_horizontal_flip(img, boxes)))
        if np.random.rand() > 0.7:
            transforms.append(('vflip', lambda img, boxes: self.apply_vertical_flip(img, boxes)))
        if np.random.rand() > 0.5:
            transforms.append(('rotate', lambda img, boxes: self.apply_rotation(img, boxes)))
        if np.random.rand() > 0.6:
            transforms.append(('translate', lambda img, boxes: self.apply_translate(img, boxes)))
        if np.random.rand() > 0.6:
            transforms.append(('brightness', lambda img, boxes: (self.apply_brightness_contrast(img), boxes)))
        if np.random.rand() > 0.7:
            transforms.append(('noise', lambda img, boxes: (self.apply_noise(img), boxes)))
        if np.random.rand() > 0.8:
            transforms.append(('blur', lambda img, boxes: (self.apply_blur(img), boxes)))
        if np.random.rand() > 0.6:
            transforms.append(('hsv', lambda img, boxes: (self.apply_hsv_shift(img), boxes)))
        if np.random.rand() > 0.7:
            transforms.append(('cutout', lambda img, boxes: (self.apply_cutout(img), boxes)))
        if np.random.rand() > 0.8:
            transforms.append(('scale', lambda img, boxes: self.apply_scale(img, boxes)))

        random.shuffle(transforms)
        num_apply = min(len(transforms), random.randint(1, 4))
        for i in range(num_apply):
            name, func = transforms[i]
            try:
                res = func(aug_img, aug_boxes)
                if isinstance(res, tuple) and len(res) == 2:          # image + boxes
                    aug_img, aug_boxes = res
                else:                                                   # single image
                    aug_img = res
                methods.append(name)
            except Exception:
                continue

        # validate
        if not isinstance(aug_img, np.ndarray):
            return None, [], [], []
        h, w = aug_img.shape[:2]
        valid_boxes, valid_labels = [], []
        for box, lab in zip(aug_boxes, aug_labels):
            if len(box) != 4:
                continue
            x1, y1, x2, y2 = self._clip_boxes([box], w, h)[0]
            if x2 - x1 > 5 and y2 - y1 > 5:
                valid_boxes.append([x1, y1, x2, y2])
                valid_labels.append(lab)
        return aug_img, valid_boxes, valid_labels, methods


# ------------------------------------------------------------------
# ---------------  top-level runner  -------------------------------
# ------------------------------------------------------------------
def run_fixed_augmentation(train_img_dir, train_xml_dir,
                           target_per_class=9000,
                           output_dir=None,
                           report_file="augmentation_report_fixed.txt"):
    """High-diversity fixed augmentation – complete version."""
    if output_dir:
        os.makedirs(os.path.join(output_dir, "images"), exist_ok=True)
        os.makedirs(os.path.join(output_dir, "annotations"), exist_ok=True)
        img_out_dir = os.path.join(output_dir, "images")
        xml_out_dir = os.path.join(output_dir, "annotations")
    else:
        img_out_dir, xml_out_dir = train_img_dir, train_xml_dir

    log_f = open(report_file, "w", encoding="utf-8")

    def log(msg):
        ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        ln = f"[{ts}] {msg}"
        print(ln)
        log_f.write(ln + "\n")
        log_f.flush()

    log("=" * 70)
    log("FIXED AUGMENTATION – COMPLETE DIVERSE PIPELINE")
    log("=" * 70)

    # 1. analyse -------------------------------------------------------------
    def analyse(xml_dir):
        cls_cnt = defaultdict(int)
        img_info = {}
        all_cls = set()
        for xml in [f for f in os.listdir(xml_dir) if f.endswith(".xml")]:
            try:
                root = ET.parse(os.path.join(xml_dir, xml)).getroot()
                boxes, labels, cls_in_img = [], [], set()
                for obj in root.findall("object"):
                    cls = obj.find("name").text
                    b = obj.find("bndbox")
                    box = [float(b.find(t).text) for t in ("xmin", "ymin", "xmax", "ymax")]
                    boxes.append(box)
                    labels.append(cls)
                    cls_in_img.add(cls)
                    cls_cnt[cls] += 1
                    all_cls.add(cls)
                img_info[xml] = {"boxes": boxes, "labels": labels, "classes": cls_in_img, "num": len(boxes)}
            except Exception as e:
                log(f"Error parsing {xml}: {e}")
        return dict(cls_cnt), img_info, sorted(all_cls)

    init_counts, img_info, all_classes = analyse(train_xml_dir)
    log("\nInitial counts:")
    for c in all_classes:
        log(f"  {c}: {init_counts[c]}")

    # 2. needs ---------------------------------------------------------------
    needs = {c: max(0, target_per_class - init_counts[c]) for c in all_classes}
    log("\nAugmentation needs:")
    for c, n in sorted(needs.items()):
        log(f"  {c}: {n}")

    # 3. schedule ------------------------------------------------------------
    log("\nCreating schedule...")
    cls2imgs = defaultdict(list)
    for xml, info in img_info.items():
        for c in info["classes"]:
            cls2imgs[c].append(xml)

    schedule = []
    for c, need in sorted(needs.items(), key=lambda x: x[1], reverse=True):
        donors = cls2imgs[c]
        if not donors:
            log(f"Warning: no donor for {c}")
            continue
        random.shuffle(donors)
        max_per_donor = min(40, max(5, int(np.ceil(need / len(donors)))))
        remaining = need
        for xml in itertools.cycle(donors):
            if remaining <= 0:
                break
            num = min(random.randint(max_per_donor - 2, max_per_donor + 2), remaining)
            schedule.append((xml, c, num))
            remaining -= num
    log(f"Planned augmentations: {len(schedule)}")

    # 4. augment -------------------------------------------------------------
    augmenter = FixedAugmenter()
    curr_counts = init_counts.copy()
    total_created = 0

    log("\nStarting augmentation...")
    for xml_file, tgt_cls, num_aug in tqdm(schedule, desc="Augmenting"):
        if all(curr_counts[c] >= target_per_class for c in all_classes):
            log("Global target reached – stopping early.")
            break
        if curr_counts[tgt_cls] >= target_per_class:
            continue

        base = os.path.splitext(xml_file)[0]
        # find image
        img_path = None
        ext = None
        for e in (".jpg", ".jpeg", ".png", ".JPG", ".PNG"):
            p = os.path.join(train_img_dir, base + e)
            if os.path.exists(p):
                img_path, ext = p, e
                break
        if not img_path:
            log(f"Image not found: {base}")
            continue
        img = cv2.imread(img_path)
        if img is None:
            log(f"Failed to load {img_path}")
            continue
        info = img_info.get(xml_file)
        if not info:
            continue
        boxes, labels = info["boxes"], info["labels"]

        for aug_idx in range(num_aug):
            if curr_counts[tgt_cls] >= target_per_class:
                break
            aug_img, aug_boxes, aug_labels, methods = augmenter.safe_augment(img, boxes, labels)
            if aug_img is None or len(aug_boxes) == 0:
                continue
            ts = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
            new_name = f"{base}_aug_{tgt_cls}_{ts}_{aug_idx}"
            img_out_path = os.path.join(img_out_dir, new_name + ext)
            try:
                if not cv2.imwrite(img_out_path, aug_img):
                    raise IOError("imwrite failed")
            except Exception as e:
                log(f"Save image error {new_name}: {e}")
                continue

            # XML
            xml_out_path = os.path.join(xml_out_dir, new_name + ".xml")
            try:
                root = ET.Element("annotation")
                for tag, text in (("folder", os.path.basename(img_out_dir)),
                                  ("filename", new_name + ext),
                                  ("path", img_out_path)):
                    ET.SubElement(root, tag).text = text
                    src = ET.SubElement(root, "source")
                ET.SubElement(src, "database").text = "Unknown"
                size = ET.SubElement(root, "size")
                for tag, val in (("width", aug_img.shape[1]), ("height", aug_img.shape[0]), ("depth", aug_img.shape[2])):
                    ET.SubElement(size, tag).text = str(val)
                ET.SubElement(root, "segmented").text = "0"
                for box, lbl in zip(aug_boxes, aug_labels):
                    obj = ET.SubElement(root, "object")
                    ET.SubElement(obj, "name").text = lbl
                    for tag in ("pose", "Unspecified"), ("truncated", "0"), ("difficult", "0"):
                        ET.SubElement(obj, tag[0]).text = tag[1]
                    bb = ET.SubElement(obj, "bndbox")
                    for tag, val in zip(("xmin", "ymin", "xmax", "ymax"), map(int, map(round, box))):
                        ET.SubElement(bb, tag).text = str(val)
                ET.ElementTree(root).write(xml_out_path, encoding="utf-8", xml_declaration=True)
            except Exception as e:
                log(f"Save XML error {new_name}: {e}")
                if os.path.exists(img_out_path):
                    os.remove(img_out_path)
                continue

            # counts
            for lbl in aug_labels:
                curr_counts[lbl] += 1
            total_created += 1

    # 5. final stats ---------------------------------------------------------
    final_counts, _, _ = analyse(xml_out_dir)
    log("\n" + "=" * 70)
    log("AUGMENTATION COMPLETE")
    log("=" * 70)
    log("Class            Init   Final   Target  Status")
    log("-" * 50)
    achieved = 0
    for c in all_classes:
        ini, fin, tgt = init_counts[c], final_counts[c], target_per_class
        status = "✓" if fin >= tgt else "✗"
        if fin >= tgt:
            achieved += 1
        log(f"{c:17}{ini:6}{fin:8}{tgt:8}  {status}")
    log(f"\nTotal created: {total_created}")
    log(f"Targets hit:   {achieved}/{len(all_classes)}")
    log_f.close()
    return total_created, final_counts


# ------------------------------------------------------------------
# ---------------  convenience wrapper  ----------------------------
# ------------------------------------------------------------------
def run_augmentation(img_input, xml_input, output_base, target=2000):
    """Full pipeline: copy originals + augment."""
    print("\n" + "=" * 70)
    print("DATASET AUGMENTATION PIPELINE – COMPLETE")
    print("=" * 70)
    os.makedirs(output_base, exist_ok=True)

    img_out = os.path.join(output_base, "images")
    xml_out = os.path.join(output_base, "annotations")
    os.makedirs(img_out, exist_ok=True)
    os.makedirs(xml_out, exist_ok=True)

    # copy / symlink originals
    def mirror(src_dir, dst_dir, exts):
        for f in os.listdir(src_dir):
            if any(f.lower().endswith(e) for e in exts):
                src = os.path.join(src_dir, f)
                dst = os.path.join(dst_dir, f)
                if not os.path.exists(dst):
                    if os.name != "nt":
                        os.symlink(src, dst)
                    else:
                        shutil.copy2(src, dst)

    mirror(img_input, img_out, [".jpg", ".jpeg", ".png"])
    mirror(xml_input, xml_out, [".xml"])

    # augment
    run_fixed_augmentation(img_out, xml_out,
                           target_per_class=target,
                           output_dir=None,  # write into same folder
                           report_file=os.path.join(output_base, "augmentation_report.txt"))
    print("\nDone – output:", output_base)
    return output_base


# ------------------------------------------------------------------
if __name__ == "1__main__":
    TRAIN_IMG = "/kaggle/working/final_dataset/train/images"
    TRAIN_XML = "/kaggle/working/final_dataset/train/annotations"
    OUT_DIR   = "/kaggle/working/final_dataset/train_aug"

    run_augmentation(TRAIN_IMG, TRAIN_XML, OUT_DIR, target=9000)

# reading dataset

In [6]:
import os
import xml.etree.ElementTree as ET

def build_class_map(xml_dir):
    class_names = set()

    for f in os.listdir(xml_dir):
        if f.endswith(".xml"):
            tree = ET.parse(os.path.join(xml_dir, f))
            root = tree.getroot()
            for obj in root.findall("object"):
                class_names.add(obj.find("name").text.strip())

    class_names = sorted(list(class_names))
    class_map = {"__background__": 0}

    for i, name in enumerate(class_names, start=1):
        class_map[name] = i

    return class_map

def parse_voc(xml_path, class_map):
    tree = ET.parse(xml_path)
    root = tree.getroot()

    boxes, labels = [], []

    for obj in root.findall("object"):
        name = obj.find("name").text.strip()
        if name not in class_map:
            continue

        b = obj.find("bndbox")
        boxes.append([
            float(b.find("xmin").text),
            float(b.find("ymin").text),
            float(b.find("xmax").text),
            float(b.find("ymax").text),
        ])
        labels.append(class_map[name])

    if len(boxes) == 0:
        return None

    return {
        "boxes": torch.tensor(boxes, dtype=torch.float32),
        "labels": torch.tensor(labels, dtype=torch.int64)
    }

def get_image_classes(xml_path, class_map):
    """Get all classes present in an image"""
    tree = ET.parse(xml_path)
    root = tree.getroot()
    
    classes = set()
    for obj in root.findall("object"):
        name = obj.find("name").text.strip()
        if name in class_map:
            classes.add(name)
    
    return list(classes)

def get_object_statistics(xml_dir, class_map):
    """Get detailed object statistics from XML files"""
    total_objects = 0
    images_with_objects = 0
    images_without_objects = 0
    class_object_counts = defaultdict(int)
    images_per_class = defaultdict(int)
    objects_per_image = []
    
    xml_files = [f for f in os.listdir(xml_dir) if f.endswith(".xml")]
    
    for xml_file in xml_files:
        xml_path = os.path.join(xml_dir, xml_file)
        tree = ET.parse(xml_path)
        root = tree.getroot()
        
        objects_in_image = 0
        classes_in_image = set()
        
        for obj in root.findall("object"):
            name = obj.find("name").text.strip()
            if name in class_map:
                total_objects += 1
                objects_in_image += 1
                class_object_counts[name] += 1
                classes_in_image.add(name)
        
        objects_per_image.append(objects_in_image)
        
        if objects_in_image > 0:
            images_with_objects += 1
            for cls in classes_in_image:
                images_per_class[cls] += 1
        else:
            images_without_objects += 1
    
    # Calculate statistics
    avg_objects_per_image = total_objects / len(xml_files) if xml_files else 0
    max_objects_per_image = max(objects_per_image) if objects_per_image else 0
    min_objects_per_image = min(objects_per_image) if objects_per_image else 0
    
    return {
        'total_objects': total_objects,
        'total_images': len(xml_files),
        'images_with_objects': images_with_objects,
        'images_without_objects': images_without_objects,
        'avg_objects_per_image': avg_objects_per_image,
        'max_objects_per_image': max_objects_per_image,
        'min_objects_per_image': min_objects_per_image,
        'class_object_counts': dict(class_object_counts),
        'images_per_class': dict(images_per_class),
        'objects_per_image_list': objects_per_image
    }

# Train Test Val Split

In [7]:
def create_stratified_splits(xml_dir, class_map, train_ratio=0.7, val_ratio=0.15, random_seed=42):
    """Create stratified splits using a simple approach"""
    # Get all XML files
    all_xml_files = sorted([f for f in os.listdir(xml_dir) if f.endswith(".xml")])
    
    # Create a simple stratification based on the presence of each class
    strat_labels = []
    
    for xml_file in all_xml_files:
        xml_path = os.path.join(xml_dir, xml_file)
        classes = get_image_classes(xml_path, class_map)
        
        if not classes:
            # Empty image
            strat_labels.append("empty")
        else:
            # Use the first class as stratification label
            strat_labels.append(classes[0])
    
    # Count occurrences of each stratification label
    from collections import Counter
    label_counts = Counter(strat_labels)
    
    # Separate files by their stratification label
    label_to_files = defaultdict(list)
    for xml_file, label in zip(all_xml_files, strat_labels):
        label_to_files[label].append(xml_file)
    
    # Initialize splits
    train_files = []
    val_files = []
    test_files = []
    
    # For each label, split its files proportionally
    for label, files in label_to_files.items():
        n_files = len(files)
        
        # Calculate number of files for each split
        n_train = max(1, int(n_files * train_ratio))
        n_val = max(1, int(n_files * val_ratio))
        n_test = n_files - n_train - n_val
        
        # Adjust if n_test is negative
        if n_test < 0:
            n_train = max(1, n_train + n_test)
            n_test = 0
        
        # Shuffle files
        random.Random(random_seed).shuffle(files)
        
        # Split
        train_files.extend(files[:n_train])
        val_files.extend(files[n_train:n_train + n_val])
        test_files.extend(files[n_train + n_val:])
    
    # Final shuffle
    random.Random(random_seed).shuffle(train_files)
    random.Random(random_seed + 1).shuffle(val_files)
    random.Random(random_seed + 2).shuffle(test_files)
    
    return train_files, val_files, test_files

def get_split_statistics(xml_dir, class_map, train_files, val_files, test_files):
    """Get detailed statistics for each split"""
    
    def get_split_stats(files):
        total_objects = 0
        class_counts = defaultdict(int)
        images_with_objects = 0
        
        for xml_file in files:
            xml_path = os.path.join(xml_dir, xml_file)
            tree = ET.parse(xml_path)
            root = tree.getroot()
            
            objects_in_image = 0
            for obj in root.findall("object"):
                name = obj.find("name").text.strip()
                if name in class_map:
                    total_objects += 1
                    objects_in_image += 1
                    class_counts[name] += 1
            
            if objects_in_image > 0:
                images_with_objects += 1
        
        return {
            'total_images': len(files),
            'total_objects': total_objects,
            'images_with_objects': images_with_objects,
            'images_without_objects': len(files) - images_with_objects,
            'avg_objects_per_image': total_objects / len(files) if files else 0,
            'class_counts': dict(class_counts)
        }
    
    train_stats = get_split_stats(train_files)
    val_stats = get_split_stats(val_files)
    test_stats = get_split_stats(test_files)
    
    return train_stats, val_stats, test_stats

In [8]:
import cv2
from torch.utils.data import Dataset

class FasterRCNNDataset(Dataset):
    def __init__(self, image_dir, xml_dir, class_map, split='train', 
                 train_files=None, val_files=None, test_files=None):
        """
        split: 'train', 'val', or 'test'
        train_files, val_files, test_files: pre-computed splits
        """
        self.image_dir = image_dir
        self.xml_dir = xml_dir
        self.class_map = class_map
        
        # Assign files based on split
        if split == 'train':
            self.xml_files = train_files
        elif split == 'val':
            self.xml_files = val_files
        elif split == 'test':
            self.xml_files = test_files
        else:
            raise ValueError("split must be 'train', 'val', or 'test'")
        
        logging.info(f"{split.upper()} dataset: {len(self.xml_files)} samples")

    def __len__(self):
        return len(self.xml_files)

    def __getitem__(self, idx):
        xml_path = os.path.join(self.xml_dir, self.xml_files[idx])
        target = parse_voc(xml_path, self.class_map)
    
        if target is None:
            return None
    
        tree = ET.parse(xml_path)
        root = tree.getroot()
        filename = root.find("filename").text
    
        img = cv2.imread(os.path.join(self.image_dir, filename))
        if img is None:
            logging.warning(f"Image not found: {filename}")
            return None
            
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
        img = torch.tensor(img, dtype=torch.float32).permute(2, 0, 1) / 255.0
        return img, target

# Model Archicture

In [9]:
import torch
import torch.nn as nn
from torchvision.models import vgg16
from torchvision.models.detection import FasterRCNN
from torchvision.models.detection.rpn import AnchorGenerator
from torchvision.ops import MultiScaleRoIAlign


def build_vgg16_faster_rcnn(num_classes, pretrained=True, freeze_backbone=True):
    # Load VGG16 backbone
    vgg = vgg16(weights="IMAGENET1K_V1" if pretrained else None)
    backbone = vgg.features

    # Freeze backbone by default
    if freeze_backbone:
        for p in backbone.parameters():
            p.requires_grad = False

    # IMPORTANT: tell FasterRCNN the output channels
    backbone.out_channels = 512

    # Anchor generator
    anchor_generator = AnchorGenerator(
        sizes=((32, 64, 128, 256, 512),),
        aspect_ratios=((0.5, 1.0, 2.0),)
    )

    # RoI pooling
    roi_pooler = MultiScaleRoIAlign(
        featmap_names=["0"],
        output_size=7,
        sampling_ratio=2
    )

    model = FasterRCNN(
        backbone=backbone,
        num_classes=num_classes,
        rpn_anchor_generator=anchor_generator,
        box_roi_pool=roi_pooler
    )

    return model


# =========================================================
# Optional: Unfreeze Backbone (VGG16)
# =========================================================
def maybe_unfreeze_backbone(model, epoch, unfreeze_epoch=10):
    if epoch == unfreeze_epoch:
        print("🔓 Unfreezing VGG16 backbone...")

        for p in model.backbone.parameters():
            p.requires_grad = True


# ---D---

In [10]:
from torch.utils.data import DataLoader

def collate_fn(batch):
    batch = [b for b in batch if b is not None]
    if len(batch) == 0:
        return [], []
    return tuple(zip(*batch))

from torchmetrics.detection.mean_ap import MeanAveragePrecision

def box_iou(box1, box2):
    area1 = (box1[:, 2] - box1[:, 0]) * (box1[:, 3] - box1[:, 1])
    area2 = (box2[:, 2] - box2[:, 0]) * (box2[:, 3] - box2[:, 1])

    lt = torch.max(box1[:, None, :2], box2[:, :2])
    rb = torch.min(box1[:, None, 2:], box2[:, 2:])

    wh = (rb - lt).clamp(min=0)
    inter = wh[:, :, 0] * wh[:, :, 1]
    union = area1[:, None] + area2 - inter

    return inter / union

def detection_metrics(
    pred_boxes,
    pred_labels,
    gt_boxes,
    gt_labels,
    iou_threshold=0.5
):
    """
    pred_boxes: Tensor[N, 4]
    pred_labels: Tensor[N]
    gt_boxes:   Tensor[M, 4]
    gt_labels: Tensor[M]
    """

    if len(pred_boxes) == 0:
        return 0.0, 0.0, 0.0

    if len(gt_boxes) == 0:
        return 0.0, 0.0, 0.0

    ious = box_iou(pred_boxes, gt_boxes)
    matched_gt = set()

    TP, FP = 0, 0

    for i in range(len(pred_boxes)):
        best_iou, gt_idx = ious[i].max(0)

        if best_iou >= iou_threshold and gt_idx.item() not in matched_gt:
            if pred_labels[i] == gt_labels[gt_idx]:
                TP += 1
                matched_gt.add(gt_idx.item())
            else:
                FP += 1
        else:
            FP += 1

    FN = len(gt_boxes) - len(matched_gt)

    precision = TP / (TP + FP + 1e-8)
    recall    = TP / (TP + FN + 1e-8)
    f1        = 2 * precision * recall / (precision + recall + 1e-8)

    return precision, recall, f1

In [11]:
# ================== MAIN TRAINING LOOP ==================
# paths
IMAGE_DIR = "/kaggle/input/pv-multi-defect-main/PV-Multi-Defect-main/JPEGImages"
XML_DIR   = "/kaggle/input/pv-multi-defect-main/PV-Multi-Defect-main/Annotations"

# build classes
CLASS_MAP = build_class_map(XML_DIR)
NUM_CLASSES = len(CLASS_MAP)

# Get detailed object statistics
logging.info("\n" + "=" * 80)
logging.info("DATASET STATISTICS")
logging.info("=" * 80)

# Get overall statistics
stats = get_object_statistics(XML_DIR, CLASS_MAP)

# Log overall statistics table
overall_stats_table = [
    ["Total Images", stats['total_images']],
    ["Images with Objects", stats['images_with_objects']],
    ["Images without Objects", stats['images_without_objects']],
    ["Total Objects", stats['total_objects']],
    ["Avg Objects per Image", f"{stats['avg_objects_per_image']:.2f}"],
    ["Max Objects per Image", stats['max_objects_per_image']],
    ["Min Objects per Image", stats['min_objects_per_image']],
    ["Images without Objects %", f"{(stats['images_without_objects']/stats['total_images']*100):.1f}%"]
]

logging.info("\nOVERALL DATASET STATISTICS:")
overall_table = tabulate(overall_stats_table, tablefmt="grid")
for line in overall_table.split('\n'):
    logging.info(line)

# Log class-wise object statistics
class_stats_table = []
for cls, count in stats['class_object_counts'].items():
    class_stats_table.append([
        cls,
        count,
        f"{(count/stats['total_objects']*100):.1f}%",
        stats['images_per_class'][cls],
        f"{(stats['images_per_class'][cls]/stats['total_images']*100):.1f}%"
    ])

logging.info("\nCLASS-WISE OBJECT STATISTICS:")
class_headers = ["Class", "Object Count", "% of Total", "Images with Class", "% of Images"]
class_table = tabulate(class_stats_table, headers=class_headers, tablefmt="grid")
for line in class_table.split('\n'):
    logging.info(line)

logging.info("=" * 80)

# Create stratified splits
logging.info("\nCreating stratified splits (70-15-15)...")
train_files, val_files, test_files = create_stratified_splits(
    XML_DIR, CLASS_MAP, 
    train_ratio=0.7, 
    val_ratio=0.15, 
    random_seed=42
)

2026-01-22 15:52:33,143 | INFO | 
2026-01-22 15:52:33,144 | INFO | DATASET STATISTICS
2026-01-22 15:52:33,145 | INFO | ================================================================================
2026-01-22 15:52:33,813 | INFO | 
OVERALL DATASET STATISTICS:
2026-01-22 15:52:33,815 | INFO | +--------------------------+------+
2026-01-22 15:52:33,816 | INFO | | Total Images             | 1106 |
2026-01-22 15:52:33,817 | INFO | +--------------------------+------+
2026-01-22 15:52:33,818 | INFO | | Images with Objects      | 1105 |
2026-01-22 15:52:33,819 | INFO | +--------------------------+------+
2026-01-22 15:52:33,820 | INFO | | Images without Objects   | 1    |
2026-01-22 15:52:33,820 | INFO | +--------------------------+------+
2026-01-22 15:52:33,821 | INFO | | Total Objects            | 3981 |
2026-01-22 15:52:33,822 | INFO | +--------------------------+------+
2026-01-22 15:52:33,823 | INFO | | Avg Objects per Image    | 3.60 |
2026-01-22 15:52:33,824 | INFO | +--------------

In [12]:
import shutil
import os

def save_organized_dataset(file_list, source_img_dir, source_xml_dir, base_target_dir, split_name, img_ext=".jpg"):
    """
    Saves images and XMLs into separate subfolders:
    base_target_dir/split_name/images/
    base_target_dir/split_name/annotations/
    """
    os.makedirs(base_target_dir, exist_ok=True)
    # Define sub-folder paths
    images_path = os.path.join(base_target_dir, split_name, "images")
    annotations_path = os.path.join(base_target_dir, split_name, "annotations")
    
    # Create folders
    os.makedirs(images_path, exist_ok=True)
    os.makedirs(annotations_path, exist_ok=True)
    
    count = 0
    for file_name in file_list:
        base_name = os.path.splitext(file_name)[0]
        
        img_src = os.path.join(source_img_dir, f"{base_name}{img_ext}")
        xml_src = os.path.join(source_xml_dir, f"{base_name}.xml")
        
        if os.path.exists(img_src) and os.path.exists(xml_src):
            # Copy to respective sub-folders
            shutil.copy(img_src, images_path)
            shutil.copy(xml_src, annotations_path)
            count += 1
            
    print(f"[{split_name.upper()}] Saved {count} images to /images and {count} XMLs to /annotations")

# --- Execute for all splits ---
save_organized_dataset(train_files, IMAGE_DIR, XML_DIR, "/kaggle/working/final_dataset", "train")
save_organized_dataset(val_files, IMAGE_DIR, XML_DIR, "/kaggle/working/final_dataset", "val")
save_organized_dataset(test_files, IMAGE_DIR, XML_DIR, "/kaggle/working/final_dataset", "test")

[TRAIN] Saved 772 images to /images and 772 XMLs to /annotations
[VAL] Saved 164 images to /images and 164 XMLs to /annotations
[TEST] Saved 170 images to /images and 170 XMLs to /annotations


In [13]:
# Get split statistics
train_stats, val_stats, test_stats = get_split_statistics(
    XML_DIR, CLASS_MAP, train_files, val_files, test_files
)

# Log split statistics in a comprehensive table
logging.info("\n" + "=" * 80)
logging.info("TRAIN-VAL-TEST SPLIT STATISTICS")
logging.info("=" * 80)

# Create comprehensive split statistics table
split_stats_table = []

# Headers
headers = ["Metric", "Train", "Validation", "Test", "Total"]

# Add rows for each metric
split_stats_table.append(["Images", 
                         f"{train_stats['total_images']} ({train_stats['total_images']/(train_stats['total_images']+val_stats['total_images']+test_stats['total_images'])*100:.1f}%)",
                         f"{val_stats['total_images']} ({val_stats['total_images']/(train_stats['total_images']+val_stats['total_images']+test_stats['total_images'])*100:.1f}%)",
                         f"{test_stats['total_images']} ({test_stats['total_images']/(train_stats['total_images']+val_stats['total_images']+test_stats['total_images'])*100:.1f}%)",
                         f"{train_stats['total_images'] + val_stats['total_images'] + test_stats['total_images']}"])

split_stats_table.append(["Objects",
                         f"{train_stats['total_objects']} ({train_stats['total_objects']/(train_stats['total_objects']+val_stats['total_objects']+test_stats['total_objects'])*100:.1f}%)",
                         f"{val_stats['total_objects']} ({val_stats['total_objects']/(train_stats['total_objects']+val_stats['total_objects']+test_stats['total_objects'])*100:.1f}%)",
                         f"{test_stats['total_objects']} ({test_stats['total_objects']/(train_stats['total_objects']+val_stats['total_objects']+test_stats['total_objects'])*100:.1f}%)",
                         f"{train_stats['total_objects'] + val_stats['total_objects'] + test_stats['total_objects']}"])

split_stats_table.append(["Avg Objects/Image",
                         f"{train_stats['avg_objects_per_image']:.2f}",
                         f"{val_stats['avg_objects_per_image']:.2f}",
                         f"{test_stats['avg_objects_per_image']:.2f}",
                         f"{stats['avg_objects_per_image']:.2f}"])

split_stats_table.append(["Images with Objects",
                         f"{train_stats['images_with_objects']} ({train_stats['images_with_objects']/train_stats['total_images']*100:.1f}%)",
                         f"{val_stats['images_with_objects']} ({val_stats['images_with_objects']/val_stats['total_images']*100:.1f}%)",
                         f"{test_stats['images_with_objects']} ({test_stats['images_with_objects']/test_stats['total_images']*100:.1f}%)",
                         f"{stats['images_with_objects']} ({stats['images_with_objects']/stats['total_images']*100:.1f}%)"])

split_stats_table.append(["Images without Objects",
                         f"{train_stats['images_without_objects']} ({train_stats['images_without_objects']/train_stats['total_images']*100:.1f}%)",
                         f"{val_stats['images_without_objects']} ({val_stats['images_without_objects']/val_stats['total_images']*100:.1f}%)",
                         f"{test_stats['images_without_objects']} ({test_stats['images_without_objects']/test_stats['total_images']*100:.1f}%)",
                         f"{stats['images_without_objects']} ({stats['images_without_objects']/stats['total_images']*100:.1f}%)"])

# Log the comprehensive split table
split_table = tabulate(split_stats_table, headers=headers, tablefmt="grid")
for line in split_table.split('\n'):
    logging.info(line)

# Log class distribution in each split
logging.info("\nCLASS DISTRIBUTION ACROSS SPLITS:")

# Get all unique classes (excluding background)
all_classes = sorted([cls for cls in CLASS_MAP.keys() if cls != "__background__"])

class_dist_table = []
for cls in all_classes:
    train_count = train_stats['class_counts'].get(cls, 0)
    val_count = val_stats['class_counts'].get(cls, 0)
    test_count = test_stats['class_counts'].get(cls, 0)
    total_count = stats['class_object_counts'].get(cls, 0)
    
    class_dist_table.append([
        cls,
        f"{train_count} ({train_count/total_count*100:.1f}%)" if total_count > 0 else "0",
        f"{val_count} ({val_count/total_count*100:.1f}%)" if total_count > 0 else "0",
        f"{test_count} ({test_count/total_count*100:.1f}%)" if total_count > 0 else "0",
        total_count
    ])

class_dist_headers = ["Class", "Train Objects", "Val Objects", "Test Objects", "Total Objects"]
class_dist_display = tabulate(class_dist_table, headers=class_dist_headers, tablefmt="grid")
for line in class_dist_display.split('\n'):
    logging.info(line)

logging.info("=" * 80)

2026-01-22 15:52:42,232 | INFO | 
2026-01-22 15:52:42,233 | INFO | TRAIN-VAL-TEST SPLIT STATISTICS
2026-01-22 15:52:42,234 | INFO | ================================================================================
2026-01-22 15:52:42,236 | INFO | +------------------------+--------------+--------------+--------------+--------------+
2026-01-22 15:52:42,237 | INFO | | Metric                 | Train        | Validation   | Test         | Total        |
2026-01-22 15:52:42,238 | INFO | +========================+==============+==============+==============+==============+
2026-01-22 15:52:42,239 | INFO | | Images                 | 772 (69.8%)  | 164 (14.8%)  | 170 (15.4%)  | 1106         |
2026-01-22 15:52:42,240 | INFO | +------------------------+--------------+--------------+--------------+--------------+
2026-01-22 15:52:42,240 | INFO | | Objects                | 2801 (70.4%) | 556 (14.0%)  | 624 (15.7%)  | 3981         |
2026-01-22 15:52:42,241 | INFO | +------------------------+--------

In [14]:
import os
import cv2
import matplotlib.pyplot as plt
from lxml import etree

def verify_saved_augmentation(img_dir, xml_dir, num_samples=10, output_filename="augmentation_verification.png"):
    # Find all augmented files
    aug_xmls = [f for f in os.listdir(xml_dir) if "_aug" in f]
    
    if not aug_xmls:
        print("No augmented files found!")
        return

    num_to_show = min(num_samples, len(aug_xmls))
    cols = 5
    rows = (num_to_show // cols) + (1 if num_to_show % cols != 0 else 0)
    
    # Increased figsize height multiplier to 6 for better clarity with 100 samples
    fig, axes = plt.subplots(rows, cols, figsize=(25, 6 * rows))
    
    if num_to_show == 1:
        axes = [axes]
    else:
        axes = axes.flatten()

    for i in range(num_to_show):
        sample_xml = aug_xmls[i]
        # Check for multiple possible extensions
        base_name = sample_xml.replace(".xml", "")
        img_path = None
        for ext in [".jpg", ".jpeg", ".png", ".JPG"]:
            temp_path = os.path.join(img_dir, base_name + ext)
            if os.path.exists(temp_path):
                img_path = temp_path
                break
        
        img = cv2.imread(img_path) if img_path else None
        
        if img is None:
            axes[i].text(0.5, 0.5, f"Missing Image:\n{base_name}", 
                         ha='center', va='center', color='red')
            axes[i].axis('off')
            continue
            
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        # Parse XML
        try:
            tree = etree.parse(os.path.join(xml_dir, sample_xml))
            for obj in tree.getroot().findall("object"):
                obj_name = obj.find("name").text
                bbox = obj.find("bndbox")
                xmin = int(bbox.find("xmin").text)
                ymin = int(bbox.find("ymin").text)
                xmax = int(bbox.find("xmax").text)
                ymax = int(bbox.find("ymax").text)
                
                # Draw Rectangle and Label
                cv2.rectangle(img, (xmin, ymin), (xmax, ymax), (255, 0, 0), 8)
                cv2.putText(img, obj_name, (xmin, ymin - 15), 
                            cv2.FONT_HERSHEY_SIMPLEX, 2.0, (255, 0, 0), 5)
        except Exception as e:
            print(f"Error parsing {sample_xml}: {e}")
        
        axes[i].imshow(img)
        axes[i].set_title(sample_xml, fontsize=12)
        axes[i].axis('off')

    # Hide unused axes
    for j in range(i + 1, len(axes)):
        axes[j].axis('off')

    plt.tight_layout()
    
    # Save the file
    plt.savefig(output_filename, dpi=100, bbox_inches='tight')
    print(f"Verification grid saved as: {output_filename}")
    plt.close() # Close figure to free up memory

# Usage:


In [15]:
run_augmentation('/kaggle/working/final_dataset/train/images', '/kaggle/working/final_dataset/train/annotations', "/kaggle/working/final_dataset/train_aug")
verify_saved_augmentation('/kaggle/working/final_dataset/train/images', '/kaggle/working/final_dataset/train/annotations',num_samples=100)


DATASET AUGMENTATION PIPELINE – COMPLETE
[2026-01-22 15:52:42] ======================================================================
[2026-01-22 15:52:42] FIXED AUGMENTATION – COMPLETE DIVERSE PIPELINE
[2026-01-22 15:52:42] ======================================================================
[2026-01-22 15:52:42] 
Initial counts:
[2026-01-22 15:52:42]   black_border: 178
[2026-01-22 15:52:42]   broken: 65
[2026-01-22 15:52:42]   hot_spot: 1456
[2026-01-22 15:52:42]   no_electricity: 126
[2026-01-22 15:52:42]   scratch: 976
[2026-01-22 15:52:42] 
Augmentation needs:
[2026-01-22 15:52:42]   black_border: 1822
[2026-01-22 15:52:42]   broken: 1935
[2026-01-22 15:52:42]   hot_spot: 544
[2026-01-22 15:52:42]   no_electricity: 1874
[2026-01-22 15:52:42]   scratch: 1024
[2026-01-22 15:52:42] 
Creating schedule...
[2026-01-22 15:52:42] Planned augmentations: 597
[2026-01-22 15:52:42] 
Starting augmentation...


Augmenting:  38%|███▊      | 225/597 [00:53<01:28,  4.21it/s]


[2026-01-22 15:53:35] Global target reached – stopping early.
[2026-01-22 15:53:36] 
[2026-01-22 15:53:36] AUGMENTATION COMPLETE
[2026-01-22 15:53:36] ======================================================================
[2026-01-22 15:53:36] Class            Init   Final   Target  Status
[2026-01-22 15:53:36] --------------------------------------------------
[2026-01-22 15:53:36] black_border        178    2000    2000  ✓
[2026-01-22 15:53:36] broken               65    2068    2000  ✓
[2026-01-22 15:53:36] hot_spot           1456    7955    2000  ✓
[2026-01-22 15:53:36] no_electricity      126    2000    2000  ✓
[2026-01-22 15:53:36] scratch             976    2345    2000  ✓
[2026-01-22 15:53:36] 
Total created: 4476
[2026-01-22 15:53:36] Targets hit:   5/5

Done – output: /kaggle/working/final_dataset/train_aug
No augmented files found!


In [16]:
verify_saved_augmentation('/kaggle/working/final_dataset/train_aug/images', '/kaggle/working/final_dataset/train_aug/annotations',num_samples=100)

Verification grid saved as: augmentation_verification.png


In [17]:
import os
import xml.etree.ElementTree as ET

def get_valid_train_files(xml_dir, img_dir, class_map):
    """
    Identifies XML/Image pairs that exist AND contain at least 
    one object from the provided class_map.
    """
    all_xml_files = sorted([f for f in os.listdir(xml_dir) if f.endswith(".xml")])
    train_files = []
    
    # Target classes from your map keys or values
    target_classes = set(class_map.keys())

    for xml_name in all_xml_files:
        base_name = os.path.splitext(xml_name)[0]
        img_name = base_name + ".jpg"
        
        src_xml = os.path.join(xml_dir, xml_name)
        src_img = os.path.join(img_dir, img_name)
        
        # 1. Check if image exists
        if os.path.exists(src_img):
            # 2. Parse XML to check for valid classes
            try:
                tree = ET.parse(src_xml)
                root = tree.getroot()
                
                # Check if any object in the XML is in our class_map
                has_valid_object = False
                for obj in root.findall('object'):
                    obj_name = obj.find('name').text
                    if obj_name in target_classes:
                        has_valid_object = True
                        break
                
                if has_valid_object:
                    train_files.append(xml_name)
            except Exception as e:
                print(f"Error reading {xml_name}: {e}")
                
    print(f"Verified {len(train_files)} files containing target classes.")
    return train_files

In [18]:
 train_files =get_valid_train_files('final_dataset/train_aug/annotations/', 'final_dataset/train_aug/images/',  CLASS_MAP)

Verified 5247 files containing target classes.


In [19]:
# Create datasets
train_dataset = FasterRCNNDataset(
    'final_dataset/train_aug/images/', "final_dataset/train_aug/annotations/", CLASS_MAP,
    split='train', 
    train_files=train_files,
    val_files=val_files,
    test_files=test_files
)

val_dataset = FasterRCNNDataset(
    IMAGE_DIR, XML_DIR, CLASS_MAP,
    split='val',
    train_files=train_files,
    val_files=val_files,
    test_files=test_files
)

test_dataset = FasterRCNNDataset(
    IMAGE_DIR, XML_DIR, CLASS_MAP,
    split='test',
    train_files=train_files,
    val_files=val_files,
    test_files=test_files
)
batch_size=8
# Create dataloaders
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=2
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=2
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=2
)

2026-01-22 15:53:53,734 | INFO | TRAIN dataset: 5247 samples
2026-01-22 15:53:53,735 | INFO | VAL dataset: 164 samples
2026-01-22 15:53:53,735 | INFO | TEST dataset: 170 samples


# model configuration

In [20]:
import torch
import logging
from tabulate import tabulate
from torchmetrics.detection import MeanAveragePrecision
import csv
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
import numpy as np
import pandas as pd
import os
from collections import Counter
# model setup
logging.info("\n" + "=" * 80)
logging.info("MODEL CONFIGURATION")
logging.info("=" * 80)

def compute_class_weights(dataset, class_map):
    counts = Counter()
    for _, targets in dataset:
        for label in targets['labels']:
            counts[label.item()] += 1

    total = sum(counts.values())
    weights = {}

    for cid, cnt in counts.items():
        weights[cid] = total / (len(counts) * cnt)

    return weights

def class_weight_dict_to_tensor(class_weight_dict, num_classes, device=None):
    """
    Convert {class_id: weight} → Tensor [w_bg, w_1, w_2, ...]
    Faster R-CNN expects background at index 0
    """
    weights = torch.ones(num_classes, dtype=torch.float)

    # background weight
    weights[0] = 1.0

    for cid, w in class_weight_dict.items():
        weights[cid] = float(w)

    if device is not None:
        weights = weights.to(device)

    return weights


class_weight_dict = compute_class_weights(train_dataset, CLASS_MAP)

class_weights = class_weight_dict_to_tensor(
    class_weight_dict,
    NUM_CLASSES,   # MUST include background
    device=device
)

logging.info("=" * 80)
logging.info(f"Class Weights Tensor: {class_weights.tolist()}")
logging.info("=" * 80)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = build_vgg16_faster_rcnn(num_classes=NUM_CLASSES, pretrained=True)


model.to(device)

# Count parameters
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())

model_info = [
    ["Device", str(device)],
    ["Total Parameters", f"{total_params:,}"],
    ["Trainable Parameters", f"{trainable_params:,}"],
    ["Trainable Percentage", f"{trainable_params/total_params*100:.1f}%"],
    ["RPN", "Trainable"],
    ["ROI Heads", "Trainable"]
]

model_table = tabulate(model_info, tablefmt="grid")
for line in model_table.split('\n'):
    logging.info(line)

logging.info("=" * 80)

# =========================================================
# Layer-wise Parameter Summary (Compact)
# =========================================================
logging.info("\n" + "=" * 80)
logging.info("LAYER-WISE PARAMETER SUMMARY")
logging.info("=" * 80)

layer_stats = []

for name, param in model.named_parameters():
    layer_stats.append([
        name,
        list(param.shape),
        f"{param.numel():,}",
        "Yes" if param.requires_grad else "No"
    ])

layer_table = tabulate(
    layer_stats,
    headers=["Layer Name", "Shape", "Params", "Trainable"],
    tablefmt="grid"
)

# Log line by line (safe for large tables)
for line in layer_table.split("\n"):
    logging.info(line)

logging.info("=" * 80)



2026-01-22 15:53:53,815 | INFO | 
2026-01-22 15:53:53,816 | INFO | MODEL CONFIGURATION
2026-01-22 15:53:53,817 | INFO | ================================================================================
2026-01-22 15:54:13,001 | INFO | ================================================================================
2026-01-22 15:54:13,003 | INFO | Class Weights Tensor: [1.0, 1.6368000507354736, 1.5829787254333496, 0.4115147590637207, 1.6368000507354736, 1.3959914445877075]
2026-01-22 15:54:13,003 | INFO | ================================================================================
Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:02<00:00, 221MB/s]


2026-01-22 15:54:17,264 | INFO | +----------------------+------------+
2026-01-22 15:54:17,265 | INFO | | Device               | cuda       |
2026-01-22 15:54:17,265 | INFO | +----------------------+------------+
2026-01-22 15:54:17,266 | INFO | | Total Parameters     | 43,884,457 |
2026-01-22 15:54:17,267 | INFO | +----------------------+------------+
2026-01-22 15:54:17,267 | INFO | | Trainable Parameters | 29,169,769 |
2026-01-22 15:54:17,268 | INFO | +----------------------+------------+
2026-01-22 15:54:17,269 | INFO | | Trainable Percentage | 66.5%      |
2026-01-22 15:54:17,270 | INFO | +----------------------+------------+
2026-01-22 15:54:17,270 | INFO | | RPN                  | Trainable  |
2026-01-22 15:54:17,271 | INFO | +----------------------+------------+
2026-01-22 15:54:17,272 | INFO | | ROI Heads            | Trainable  |
2026-01-22 15:54:17,273 | INFO | +----------------------+------------+
2026-01-22 15:54:17,274 | INFO | ============================================

In [21]:
CLASS_MAP

{'__background__': 0,
 'black_border': 1,
 'broken': 2,
 'hot_spot': 3,
 'no_electricity': 4,
 'scratch': 5}

In [22]:

def compute_precision_recall_curve(preds, gts, target_class_idx, iou_threshold=0.5):
    """
    Compute precision and recall points for a single class across the entire dataset.
    
    Args:
        preds: List of prediction dicts {'boxes', 'scores', 'labels'}
        gts:   List of ground truth dicts {'boxes', 'labels'}
        target_class_idx: Integer class ID to evaluate
        iou_threshold: IoU threshold for TP (default 0.5)
    
    Returns:
        precision_points: list of precision values (at different recall levels)
        recall_points:    list of corresponding recall values
    """
    # Collect all predictions and ground truths for the target class
    all_scores = []
    all_is_tp = []
    num_gts = 0

    for pred, gt in zip(preds, gts):
        # Ground truth boxes/labels for target class
        gt_mask = gt['labels'] == target_class_idx
        gt_boxes = gt['boxes'][gt_mask]
        num_gts += gt_boxes.shape[0]

        # Prediction boxes/labels/scores for target class
        pred_mask = pred['labels'] == target_class_idx
        if pred_mask.sum() == 0:
            continue  # no predictions for this class in this image

        pred_boxes = pred['boxes'][pred_mask]
        pred_scores = pred['scores'][pred_mask]

        if gt_boxes.shape[0] == 0:
            # No ground truth → all predictions are FP
            all_scores.extend(pred_scores.tolist())
            all_is_tp.extend([0] * len(pred_scores))
            continue

        # Compute IoU between predictions and ground truths
        ious = box_iou(pred_boxes, gt_boxes)  # shape: (n_pred, n_gt)

        # Assign each prediction to at most one GT (greedy matching by score order later)
        # We'll sort predictions by decreasing score globally
        all_scores.extend(pred_scores.tolist())

        # For each prediction, check if it matches any GT with IoU >= threshold
        matched_gt = torch.zeros(gt_boxes.shape[0], dtype=torch.bool, device=ious.device)
        tp_flags = torch.zeros(pred_boxes.shape[0], dtype=torch.bool, device=ious.device)

        # Sort predictions by score descending for proper ranking
        sorted_idx = torch.argsort(pred_scores, descending=True)
        for idx in sorted_idx:
            if tp_flags[idx]:  # already marked as TP
                continue
            iou_row = ious[idx]
            best_iou, best_gt_idx = iou_row.max(dim=0)
            if best_iou >= iou_threshold and not matched_gt[best_gt_idx]:
                tp_flags[idx] = True
                matched_gt[best_gt_idx] = True

        all_is_tp.extend(tp_flags.cpu().tolist())

    if not all_scores:
        return [], []

    # Sort all predictions by decreasing confidence score
    sorted_indices = np.argsort(all_scores)[::-1]
    all_is_tp = np.array(all_is_tp)[sorted_indices]

    # Cumulative TP and FP
    tp_cum = np.cumsum(all_is_tp)
    fp_cum = np.cumsum(~all_is_tp)
    total_pred = len(all_scores)

    # Avoid division by zero
    eps = 1e-8
    recalls = tp_cum / (num_gts + eps)
    precisions = tp_cum / (tp_cum + fp_cum + eps)

    # Return as lists of points (one per prediction)
    return precisions.tolist(), recalls.tolist()


def save_confidence_matrix_to_csv(count_matrix, conf_matrix, iou_matrix, class_names, filepath):
    """Save confidence matrix data to CSV"""
    with open(filepath, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['actual_class', 'predicted_class', 'count', 'avg_confidence', 'avg_iou'])
        
        for i, actual_class in enumerate(class_names):
            for j, pred_class in enumerate(class_names):
                writer.writerow([
                    actual_class,
                    pred_class,
                    int(count_matrix[i, j]),
                    f"{conf_matrix[i, j]:.4f}",
                    f"{iou_matrix[i, j]:.4f}"
                ])



def save_essential_metrics(epoch, epoch_metrics, classwise_metrics, per_class_prf1, class_weights):
    """
    Save essential metrics to CSV files
    """
    # 1. Save main metrics
    with open(os.path.join(config.output_root, "metrics/training_metrics.csv"), 'a', newline='') as f:
        writer = csv.writer(f)
        writer.writerow([
            epoch,
            epoch_metrics['train_loss'],
            epoch_metrics['train_weighted_loss'],
            epoch_metrics['val_precision'],
            epoch_metrics['val_recall'],
            epoch_metrics['val_f1'],
            epoch_metrics['val_map_05'],
            epoch_metrics['val_weighted_map'],
            epoch_metrics['best_epoch_flag']
        ])
    
    # 2. Save class-wise mAP@0.5 with weights
    with open(os.path.join(config.output_root, "metrics/classwise_map_05.csv"), 'a', newline='') as f:
        writer = csv.writer(f)
        for class_name, map_05 in classwise_metrics.items():
            class_id = CLASS_MAP[class_name]
            weight = class_weights.get(class_id, 1.0)
            writer.writerow([
                epoch,
                class_name,
                class_id,
                map_05,
                weight
            ])
    
    # 3. Save per-class precision/recall/F1 with weights
    with open(os.path.join(config.output_root, "metrics/classwise_prf1.csv"), 'a', newline='') as f:
        writer = csv.writer(f)
        for class_name, metrics in per_class_prf1.items():
            class_id = CLASS_MAP[class_name]
            weight = class_weights.get(class_id, 1.0)
            writer.writerow([
                epoch,
                class_name,
                class_id,
                metrics.get('precision', 0),
                metrics.get('recall', 0),
                metrics.get('f1', 0),
                weight
            ])
    
    # 4. Save per-epoch metrics to separate file
    epoch_metrics_file = os.path.join(config.output_root, f"metrics/per_epoch/epoch_{epoch}_metrics.csv")
    with open(epoch_metrics_file, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['metric', 'value'])
        for key, value in epoch_metrics.items():
            writer.writerow([key, value])
        
        writer.writerow([])
        writer.writerow(['class', 'map_05', 'precision', 'recall', 'f1', 'weight'])
        for class_name in NON_BACKGROUND_CLASS_NAMES:
            writer.writerow([
                class_name,
                classwise_metrics.get(class_name, 0),
                per_class_prf1.get(class_name, {}).get('precision', 0),
                per_class_prf1.get(class_name, {}).get('recall', 0),
                per_class_prf1.get(class_name, {}).get('f1', 0),
                class_weights.get(CLASS_MAP[class_name], 1.0)
            ])

# Class Definitions

In [23]:
CLASS_MAP = {
    '__background__': 0,
    'black_border': 1,
    'broken': 2,
    'hot_spot': 3,
    'no_electricity': 4,
    'scratch': 5
}

ALL_CLASS_NAMES = list(CLASS_MAP.keys())
BACKGROUND_CLASS_NAME = '__background__'
BACKGROUND_CLASS_ID = CLASS_MAP[BACKGROUND_CLASS_NAME]

NON_BACKGROUND_CLASS_NAMES = [
    c for c in ALL_CLASS_NAMES if c != BACKGROUND_CLASS_NAME
]

# Training Configuration

In [24]:
class TrainingConfig:
    def __init__(self):
        self.output_root = "output"
        self.num_epochs = 100
        self.learning_rate = 1e-4
        self.weight_decay = 1e-4
        self.confidence_thresholds = [0.0]
        self.batch_size =batch_size

        self.create_directories()

    def create_directories(self):
        subdirs = [
            "metrics",
            "models",
            "plots",
            "logs",
            "test_results"
        ]
        for d in subdirs:
            os.makedirs(os.path.join(self.output_root, d), exist_ok=True)

config = TrainingConfig()

# CSV init

In [25]:
# 1. Main training metrics CSV
csv_file = os.path.join(config.output_root, "metrics/training_metrics.csv")
with open(csv_file, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow([
        'epoch', 'train_loss',
        'val_precision', 'val_recall', 'val_f1',
        'val_map_05', 'best_epoch'
    ])
# 2. Class-wise mAP@0.5 CSV
class_map_csv_file = os.path.join(config.output_root, "metrics/classwise_map_05.csv")
with open(class_map_csv_file, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['epoch', 'class_name', 'class_id', 'map_05', 'weight'])

# 3. Per-class precision, recall, F1 CSV
class_prf1_csv = os.path.join(config.output_root, "metrics/classwise_prf1.csv")
with open(class_prf1_csv, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['epoch', 'class_name', 'class_id', 'precision', 'recall', 'f1', 'weight'])


# Model, Optimizer, Metrics

In [26]:
model.to(device)

params = [p for p in model.parameters() if p.requires_grad]

optimizer = torch.optim.AdamW(
    params,
    lr=config.learning_rate,
    weight_decay=config.weight_decay
)

map_metric_05 = MeanAveragePrecision(
    iou_thresholds=[0.5],
    class_metrics=True
)

# Training Loop and Validation Loop

In [27]:
import torchvision
import sys

def build_checkpoint_metadata(config):
    return {
        "class_map": CLASS_MAP,                      # name -> id
        "id_to_class_name": ID_TO_CLASS_NAME,        # id -> name
        "num_classes": len(CLASS_MAP),
        "background_class_id": BACKGROUND_CLASS_ID,

        "metrics": {
            "map_iou_thresholds": [0.1, 0.3, 0.5],
            "primary_selection_metric": "mAP@0.5",
            "early_stopping_metric": "mAP@0.3",
        },

        "training_config": {
            "num_epochs": config.num_epochs,
            "batch_size": config.batch_size,
            "optimizer": optimizer.__class__.__name__,
            "initial_lr": optimizer.param_groups[0]["lr"],
            "early_stop_patience": early_stop_patience,
        },

        "environment": {
            "python_version": sys.version,
            "torch_version": torch.__version__,
            "torchvision_version": torchvision.__version__,
            "device": str(device),
        }
    }


In [28]:
# =========================================================
# Imports
# =========================================================
import os
import csv
import logging
import numpy as np
import matplotlib.pyplot as plt
import torch
from torchvision.ops import box_iou
from torchmetrics.detection.mean_ap import MeanAveragePrecision

# =========================================================
# Metrics (ONLY mAP@0.5)
# =========================================================
ID_TO_CLASS_NAME = {v: k for k, v in CLASS_MAP.items()}

map_metric = MeanAveragePrecision(
    iou_thresholds=[0.5],
    box_format="xyxy",
    class_metrics=False
)

map_metric_classwise = MeanAveragePrecision(
    iou_thresholds=[0.5],
    box_format="xyxy",
    class_metrics=True
)

# =========================================================
# Early stopping & best model tracking (mAP@0.5 ONLY)
# =========================================================
early_stop_counter = 0
early_stop_patience = 15
min_delta = 1e-4

best_map_05 = 0.0
best_epoch_05 = 0

# =========================================================
# Output dirs
# =========================================================
metrics_dir = os.path.join(config.output_root, "metrics")
pr_dir = os.path.join(metrics_dir, "pr_curves")
models_dir = os.path.join(config.output_root, "models")

os.makedirs(metrics_dir, exist_ok=True)
os.makedirs(pr_dir, exist_ok=True)
os.makedirs(models_dir, exist_ok=True)

best_model_path = os.path.join(models_dir, "best_model.pth")
final_model_path = os.path.join(models_dir, "final_model.pth")

# =========================================================
# CSV init (class-wise AP@0.5)
# =========================================================
class_map_csv = os.path.join(metrics_dir, "classwise_map_05.csv")
with open(class_map_csv, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["epoch", "class_name", "class_id", "ap_05"])

# =========================================================
# Helper functions
# =========================================================
def filter_predictions_only(preds, gts):
    filtered_preds = []
    for pred in preds:
        mask = pred["labels"] != BACKGROUND_CLASS_ID
        filtered_preds.append({
            "boxes": pred["boxes"][mask],
            "scores": pred["scores"][mask],
            "labels": pred["labels"][mask],
        })
    return filtered_preds, gts


def safe_extract_map_value(map_result, key):
    if key not in map_result or map_result[key] is None:
        return 0.0
    return max(map_result[key].item(), 0.0)


# =========================================================
# PR Curve (IoU = 0.5)
# =========================================================
def compute_pr_curve_single_class(preds, gts, class_id, iou_thresh=0.5):
    scores, tps, fps = [], [], []
    total_gt = 0

    for pred, gt in zip(preds, gts):
        p_mask = pred["labels"] == class_id
        g_mask = gt["labels"] == class_id

        p_boxes = pred["boxes"][p_mask]
        p_scores = pred["scores"][p_mask]
        g_boxes = gt["boxes"][g_mask]

        total_gt += len(g_boxes)
        if len(p_boxes) == 0:
            continue

        order = torch.argsort(p_scores, descending=True)
        p_boxes = p_boxes[order]
        p_scores = p_scores[order]

        matched = torch.zeros(len(g_boxes))
        ious = box_iou(p_boxes, g_boxes) if len(g_boxes) > 0 else None

        for i in range(len(p_boxes)):
            if ious is None:
                fps.append(1)
                tps.append(0)
            else:
                max_iou, idx = ious[i].max(0)
                if max_iou >= iou_thresh and matched[idx] == 0:
                    tps.append(1)
                    fps.append(0)
                    matched[idx] = 1
                else:
                    tps.append(0)
                    fps.append(1)

            scores.append(p_scores[i].item())

    if total_gt == 0 or len(tps) == 0:
        return None, None

    tps = np.cumsum(tps)
    fps = np.cumsum(fps)

    recall = tps / (total_gt + 1e-6)
    precision = tps / (tps + fps + 1e-6)

    return recall, precision


def plot_combined_pr_curve(preds, gts, epoch, save_dir, id_to_class_name, map_05):
    plt.figure(figsize=(9, 7))
    all_precisions = []

    for cid in sorted(id_to_class_name.keys()):
        if cid == BACKGROUND_CLASS_ID:
            continue

        recall, precision = compute_pr_curve_single_class(
            preds, gts, cid, iou_thresh=0.5
        )

        if recall is None:
            continue

        all_precisions.append(precision)
        plt.plot(recall, precision, linewidth=1.5,
                 label=id_to_class_name.get(cid, f"class_{cid}"))

    if all_precisions:
        min_len = min(len(p) for p in all_precisions)
        mean_precision = np.mean([p[:min_len] for p in all_precisions], axis=0)
        mean_recall = np.linspace(0, 1, min_len)

        plt.plot(mean_recall, mean_precision, linewidth=3,
                 label=f"mAP@0.5 = {map_05:.3f}")

    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title("Precision–Recall Curve (IoU=0.5)")
    plt.xlim(0, 1)
    plt.ylim(0, 1)
    plt.grid(True)
    plt.legend()
    plt.tight_layout()

    plt.savefig(os.path.join(save_dir, f"epoch_{epoch:03d}_pr_curve.png"), dpi=300)
    plt.close()


# =========================================================
# Training loop
# =========================================================
logging.info("🚀 Start Training")

for epoch in range(config.num_epochs):

    maybe_unfreeze_backbone(model, epoch, unfreeze_epoch=10)

    if epoch == 10:
        for g in optimizer.param_groups:
            g["lr"] = 1e-4

    # ------------------ TRAIN ------------------
    model.train()
    epoch_loss = 0.0

    for images, targets in train_loader:
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        loss_dict = model(images, targets)
        loss = sum(loss_dict.values())

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    logging.info(f"[Epoch {epoch+1}] Train Loss: {epoch_loss:.4f}")

    # ------------------ VALIDATION ------------------
    model.eval()
    map_metric.reset()
    map_metric_classwise.reset()

    all_preds, all_gts = [], []

    with torch.no_grad():
        for images, targets in val_loader:
            images = [img.to(device) for img in images]
            outputs = model(images)

            preds, gts = [], []
            for out, tgt in zip(outputs, targets):
                preds.append({
                    "boxes": out["boxes"].cpu(),
                    "scores": out["scores"].cpu(),
                    "labels": out["labels"].cpu()
                })
                gts.append({
                    "boxes": tgt["boxes"].cpu(),
                    "labels": tgt["labels"].cpu()
                })

            preds, gts = filter_predictions_only(preds, gts)

            valid = [(p, g) for p, g in zip(preds, gts) if len(g["boxes"]) > 0]
            if valid:
                p, g = zip(*valid)
                map_metric.update(list(p), list(g))
                map_metric_classwise.update(list(p), list(g))
                all_preds.extend(p)
                all_gts.extend(g)

    # ------------------ METRICS ------------------
    map_res = map_metric.compute()
    map_res_class = map_metric_classwise.compute()

    val_map_05 = safe_extract_map_value(map_res, "map_50")

    # ------------------ CSV ------------------
    with open(class_map_csv, "a", newline="") as f:
        writer = csv.writer(f)
        for cid, ap in zip(
            map_res_class["classes"].cpu().numpy(),
            map_res_class["map_per_class"].cpu().numpy()
        ):
            writer.writerow([
                epoch + 1,
                ID_TO_CLASS_NAME.get(int(cid), f"class_{cid}"),
                int(cid),
                round(float(ap), 6)
            ])

    # ------------------ PR CURVE ------------------
    plot_combined_pr_curve(
        all_preds, all_gts, epoch + 1, pr_dir, ID_TO_CLASS_NAME, val_map_05
    )

    # ------------------ EARLY STOPPING (mAP@0.5) ------------------
    if val_map_05 > best_map_05 + min_delta:
        best_map_05 = val_map_05
        best_epoch_05 = epoch + 1
        early_stop_counter = 0

        torch.save(
            {
                "epoch": epoch + 1,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "map_05": best_map_05,
                "metadata": build_checkpoint_metadata(config)
            },
            best_model_path
        )

        logging.info(f"💾 Saved BEST model @ epoch {epoch+1} (mAP@0.5={best_map_05:.4f})")

    else:
        early_stop_counter += 1

    logging.info(f"[Epoch {epoch+1}] Val mAP@0.5: {val_map_05:.4f}")

    if early_stop_counter >= early_stop_patience:
        logging.info(
            f"🛑 Early stopping @ epoch {epoch+1} "
            f"(best mAP@0.5={best_map_05:.4f} @ epoch {best_epoch_05})"
        )
        break

# =========================================================
# Save FINAL model
# =========================================================
torch.save(
    {
        "epoch": epoch + 1,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "map_05": val_map_05,
        "metadata": build_checkpoint_metadata(config)
    },
    final_model_path
)

logging.info(f"📦 Final model saved to {final_model_path}")
print(f"Training finished | Best mAP@0.5 = {best_map_05:.4f} (epoch {best_epoch_05})")


2026-01-22 15:54:17,997 | INFO | 🚀 Start Training
2026-01-22 15:59:23,691 | INFO | [Epoch 1] Train Loss: 181.2953
2026-01-22 15:59:35,029 | INFO | 💾 Saved BEST model @ epoch 1 (mAP@0.5=0.4726)
2026-01-22 15:59:35,031 | INFO | [Epoch 1] Val mAP@0.5: 0.4726
2026-01-22 16:04:40,640 | INFO | [Epoch 2] Train Loss: 124.6301
2026-01-22 16:04:52,580 | INFO | 💾 Saved BEST model @ epoch 2 (mAP@0.5=0.5164)
2026-01-22 16:04:52,581 | INFO | [Epoch 2] Val mAP@0.5: 0.5164
2026-01-22 16:09:58,348 | INFO | [Epoch 3] Train Loss: 110.5217
2026-01-22 16:10:10,458 | INFO | 💾 Saved BEST model @ epoch 3 (mAP@0.5=0.5376)
2026-01-22 16:10:10,459 | INFO | [Epoch 3] Val mAP@0.5: 0.5376
2026-01-22 16:15:17,593 | INFO | [Epoch 4] Train Loss: 100.3843
2026-01-22 16:15:29,518 | INFO | 💾 Saved BEST model @ epoch 4 (mAP@0.5=0.5655)
2026-01-22 16:15:29,520 | INFO | [Epoch 4] Val mAP@0.5: 0.5655
2026-01-22 16:20:36,467 | INFO | [Epoch 5] Train Loss: 93.0410
2026-01-22 16:20:48,686 | INFO | 💾 Saved BEST model @ epoch 5 (

# Testing

In [29]:
# =========================================================
# Imports
# =========================================================
import os
import csv
import logging
import numpy as np
import matplotlib.pyplot as plt
import torch

from torchvision.ops import box_iou
from torchmetrics.detection.mean_ap import MeanAveragePrecision


# =========================================================
# Class helpers
# =========================================================
ID_TO_CLASS_NAME = {v: k for k, v in CLASS_MAP.items()}
NUM_CLASSES = len(ID_TO_CLASS_NAME)


# =========================================================
# SINGLE TEST DIRECTORY
# =========================================================
test_dir = os.path.join("/kaggle/working/", "test")
os.makedirs(test_dir, exist_ok=True)


# =========================================================
# Metric
# =========================================================
test_map_metric = MeanAveragePrecision(
    iou_thresholds=[0.5],
    box_format="xyxy",
    class_metrics=True
)


# =========================================================
# Helpers
# =========================================================
def filter_predictions_only(preds, gts):
    filtered_preds = []
    for p in preds:
        mask = p["labels"] != BACKGROUND_CLASS_ID
        filtered_preds.append({
            "boxes": p["boxes"][mask],
            "scores": p["scores"][mask],
            "labels": p["labels"][mask],
        })
    return filtered_preds, gts


def safe_extract_map_value(res, key):
    if key not in res or res[key] is None:
        return 0.0
    try:
        return max(res[key].item(), 0.0)
    except Exception:
        return 0.0


# =========================================================
# Precision–Recall Curve (per class)
# =========================================================
def compute_pr_curve_single_class(preds, gts, class_id, iou_thresh=0.5):
    tps, fps = [], []
    total_gt = 0

    for pred, gt in zip(preds, gts):
        p_mask = pred["labels"] == class_id
        g_mask = gt["labels"] == class_id

        p_boxes = pred["boxes"][p_mask]
        p_scores = pred["scores"][p_mask]
        g_boxes = gt["boxes"][g_mask]

        total_gt += len(g_boxes)
        if len(p_boxes) == 0:
            continue

        order = torch.argsort(p_scores, descending=True)
        p_boxes = p_boxes[order]
        p_scores = p_scores[order]

        matched = torch.zeros(len(g_boxes))
        ious = box_iou(p_boxes, g_boxes) if len(g_boxes) > 0 else None

        for i in range(len(p_boxes)):
            if ious is None:
                tps.append(0)
                fps.append(1)
            else:
                max_iou, idx = ious[i].max(0)
                if max_iou >= iou_thresh and matched[idx] == 0:
                    tps.append(1)
                    fps.append(0)
                    matched[idx] = 1
                else:
                    tps.append(0)
                    fps.append(1)

    if total_gt == 0 or len(tps) == 0:
        return None, None

    tps = np.cumsum(tps)
    fps = np.cumsum(fps)

    recall = tps / (total_gt + 1e-6)
    precision = tps / (tps + fps + 1e-6)

    return recall, precision


# =========================================================
# Plot PR Curves (NO mean mAP)
# =========================================================
def plot_combined_pr_curve(preds, gts, save_path):
    plt.figure(figsize=(9, 7))

    for cid, name in ID_TO_CLASS_NAME.items():
        if cid == BACKGROUND_CLASS_ID:
            continue

        recall, precision = compute_pr_curve_single_class(
            preds, gts, cid, iou_thresh=0.5
        )

        if recall is None:
            continue

        plt.plot(recall, precision, linewidth=1.5, label=name)

    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title("Precision–Recall Curve (IoU = 0.5)")
    plt.xlim(0, 1)
    plt.ylim(0, 1)
    plt.grid(True)
    plt.legend(loc="lower left")
    plt.tight_layout()
    plt.savefig(save_path, dpi=300)
    plt.close()


# =========================================================
# Confusion Matrix (IoU-based detection)
# =========================================================
def compute_detection_confusion_matrix(preds, gts, iou_thresh=0.5):
    cm = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=np.int64)

    for pred, gt in zip(preds, gts):
        p_boxes = pred["boxes"]
        p_labels = pred["labels"]
        g_boxes = gt["boxes"]
        g_labels = gt["labels"]

        if len(g_boxes) == 0 or len(p_boxes) == 0:
            continue

        matched_gt = np.zeros(len(g_boxes))
        ious = box_iou(p_boxes, g_boxes)

        for i in range(len(p_boxes)):
            max_iou, idx = ious[i].max(0)
            if max_iou >= iou_thresh and matched_gt[idx] == 0:
                gt_cls = int(g_labels[idx])
                pr_cls = int(p_labels[i])
                cm[gt_cls, pr_cls] += 1
                matched_gt[idx] = 1

    return cm


def plot_confusion_matrix(cm, save_path, normalize=True):
    if normalize:
        cm = cm.astype(np.float32)
        cm = cm / (cm.sum(axis=1, keepdims=True) + 1e-6)

    plt.figure(figsize=(8, 7))
    plt.imshow(cm, cmap="Blues")  # 🔵 WHITE → BLUE
    plt.title("Normalized Detection Confusion Matrix (IoU ≥ 0.5)")
    plt.colorbar()

    classes = [ID_TO_CLASS_NAME[i] for i in range(NUM_CLASSES)]
    ticks = np.arange(NUM_CLASSES)
    plt.xticks(ticks, classes, rotation=45, ha="right")
    plt.yticks(ticks, classes)

    thresh = cm.max() / 2.0
    for i in range(NUM_CLASSES):
        for j in range(NUM_CLASSES):
            plt.text(
                j, i, f"{cm[i, j]:.2f}",
                ha="center", va="center",
                color="white" if cm[i, j] > thresh else "black"
            )

    plt.xlabel("Predicted")
    plt.ylabel("Ground Truth")
    plt.tight_layout()
    plt.savefig(save_path, dpi=300)
    plt.close()


# =========================================================
# TEST LOOP
# =========================================================
logging.info("🧪 Running Test Evaluation")

model.eval()
test_map_metric.reset()

all_preds, all_gts = [], []

with torch.no_grad():
    for images, targets in test_loader:
        images = [img.to(device) for img in images]
        outputs = model(images)

        preds, gts = [], []
        for out, tgt in zip(outputs, targets):
            preds.append({
                "boxes": out["boxes"].cpu(),
                "scores": out["scores"].cpu(),
                "labels": out["labels"].cpu()
            })
            gts.append({
                "boxes": tgt["boxes"].cpu(),
                "labels": tgt["labels"].cpu()
            })

        preds, gts = filter_predictions_only(preds, gts)

        valid = [(p, g) for p, g in zip(preds, gts) if len(g["boxes"]) > 0]
        if valid:
            p_valid, g_valid = zip(*valid)
            test_map_metric.update(list(p_valid), list(g_valid))
            all_preds.extend(p_valid)
            all_gts.extend(g_valid)


# =========================================================
# METRICS + SAVING
# =========================================================
res = test_map_metric.compute()
map_50 = safe_extract_map_value(res, "map_50")

# ---- CSV
csv_path = os.path.join(test_dir, "classwise_map_05.csv")
with open(csv_path, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["class_name", "class_id", "map_05"])
    for cid, ap in zip(
        res["classes"].cpu().numpy(),
        res["map_per_class"].cpu().numpy()
    ):
        writer.writerow([
            ID_TO_CLASS_NAME[int(cid)],
            int(cid),
            round(float(ap), 6)
        ])

# ---- PR curve
plot_combined_pr_curve(
    all_preds,
    all_gts,
    save_path=os.path.join(test_dir, "precision_recall_curve.png")
)

# ---- Normalized confusion matrix (WHITE → BLUE)
cm = compute_detection_confusion_matrix(all_preds, all_gts)
plot_confusion_matrix(
    cm,
    save_path=os.path.join(test_dir, "confusion_matrix.png"),
    normalize=True
)

logging.info(f"✅ Test completed | mAP@0.5 = {map_50:.4f}")

2026-01-22 20:17:11,787 | INFO | 🧪 Running Test Evaluation
2026-01-22 20:17:23,923 | INFO | ✅ Test completed | mAP@0.5 = 0.5298


In [30]:
import os
import shutil

root_path = '/kaggle/working/'
os.chdir(root_path)

files_in_root = []

for item in os.listdir('.'):
    if item.startswith('.'):
        continue
    
    if os.path.isdir(item):
        # 1. Zip the directory
        shutil.make_archive(item, 'zip', item)
        # 2. Delete the original directory
        shutil.rmtree(item)
        print(f"Zipped and deleted folder: {item}")
        
    elif os.path.isfile(item):
        # --- UPDATED CODE START ---
        # Skip existing zips AND skip .pth files
        if not item.endswith('.zip') and not item.endswith('.pth'):
            files_in_root.append(item)
        # --- UPDATED CODE END ---

# 3. Handle root files: Zip them into root.zip if any exist
if files_in_root:
    temp_dir = 'root_files_temp'
    os.makedirs(temp_dir, exist_ok=True)
    
    for f in files_in_root:
        shutil.move(f, os.path.join(temp_dir, f))
    
    shutil.make_archive('root', 'zip', temp_dir)
    shutil.rmtree(temp_dir)
    print("Zipped and deleted root level files into root.zip (excluding .pth files)")

Zipped and deleted folder: output
Zipped and deleted folder: final_dataset
Zipped and deleted folder: logs
Zipped and deleted folder: test
Zipped and deleted root level files into root.zip (excluding .pth files)
